# Digit Recognition using CNN

**Project:** Digit-Recognition  
**Dataset:** Kaggle Digit Recognizer (28×28 grayscale handwritten digits)  
**Goal:** Classify handwritten digits from 0 to 9 using a Convolutional Neural Network (CNN).

Run the cells from top to bottom. Keep the `data` folder beside this notebook's project folder structure.

In [ ]:
# STEP 1: Import libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)
print("Project folder:", Path.cwd())

In [ ]:
# STEP 2: Load the dataset (relative paths work on another computer too)
DATA_DIR = Path("../data") if not Path("data").exists() else Path("data")
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Dataset files not found. Keep train.csv and test.csv inside the project's data folder."
    )

df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training data shape:", df.shape)
print("Test data shape:", test_df.shape)
display(df.head())

In [ ]:
# STEP 3: Preprocess the data
y = df["label"].to_numpy()
X = df.drop(columns=["label"]).to_numpy(dtype="float32")

# Normalize pixel values from 0-255 to 0-1
X = X / 255.0

# Reshape each image from 784 pixels to 28x28 with one grayscale channel
X = X.reshape(-1, 28, 28, 1)

print("Images:", X.shape)
print("Labels:", y.shape)
print("Classes:", np.unique(y))

In [ ]:
# STEP 4: Split training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)

In [ ]:
# STEP 5: Display sample handwritten digits
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# STEP 6: Build the Convolutional Neural Network (CNN)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.summary()

In [ ]:
# STEP 7: Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled successfully.")

In [ ]:
# STEP 8: Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
# STEP 9: Plot training and validation accuracy/loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# STEP 10: Evaluate model and show classification report
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")

val_probabilities = model.predict(X_val, verbose=0)
val_predictions = np.argmax(val_probabilities, axis=1)
print(classification_report(y_val, val_predictions))

In [ ]:
# STEP 11: Display confusion matrix
cm = confusion_matrix(y_val, val_predictions)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# STEP 12: Predict a digit from the provided test.csv file
X_test = test_df.to_numpy(dtype="float32") / 255.0
X_test = X_test.reshape(-1, 28, 28, 1)

idx = 5  # Change this number to inspect another test image
if idx >= len(X_test):
    raise IndexError("Selected image index is outside the test dataset.")

probabilities = model.predict(X_test[idx:idx + 1], verbose=0)
predicted_digit = int(np.argmax(probabilities))

plt.imshow(X_test[idx].squeeze(), cmap="gray")
plt.title(f"Predicted digit: {predicted_digit}")
plt.axis("off")
plt.show()
print("Predicted digit:", predicted_digit)

In [ ]:
# STEP 13: Save the trained model and test predictions
model.save("digit_recognition_model.keras")

test_probabilities = model.predict(X_test, batch_size=256, verbose=1)
test_predictions = np.argmax(test_probabilities, axis=1)
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(test_predictions) + 1),
    "Label": test_predictions
})
submission.to_csv("submission.csv", index=False)

print("Saved model: digit_recognition_model.keras")
print("Saved predictions: submission.csv")
display(submission.head())